# Group-Level FC Analysis — Permutation-Based Statistics
## Multi-Subject Workflow Tutorial

**Goal:** Aggregate individual FC matrices across runs and sessions, then test
group-level hypotheses using a GLM framework with permutation-based multiple
comparison correction.

```
Raw run-level FC CSVs
        │
        ▼ aggregate_subject_runs()     ← Section 1
Subject-level FC CSVs
        │
        ▼ load_fc_from_file_list()     ← Section 2
        │  load_design_matrix()
        ▼ permutation_glm()
t-maps + null distribution
        │
        ▼ plot_corrected_maps()        ← Section 3
Uncorrected | FDR | FWE maps + edge CSV
```

---
**Helper file split:**

| File | Contains |
|------|----------|
| `fc_helpers.py` | Single-subject functions: `compute_fc`, `plot_fc_matrix`, `save_fc_matrix`, `load_fc_matrix`, `fc_help` |
| `group_fc_helpers.py` | Group functions: `aggregate_subject_runs`, `load_fc_from_file_list`, `load_design_matrix`, `permutation_glm`, `plot_corrected_maps`, `save_edge_results`, `group_fc_help` |

---
**Prerequisites:** Complete the FC tutorial and run `save_fc_matrix()` for each subject.

## Imports

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from fc_helpers import plot_fc_matrix
from group_fc_helpers import *

# nice for testing / debugging helper functions
import importlib, group_fc_helpers
importlib.reload(group_fc_helpers)

from group_fc_helpers import *

warnings.filterwarnings('ignore')
%matplotlib inline
print('Imports OK')
group_fc_help()

---
# Section 1 — Prepare Your Data

## Why aggregate runs first?

Many protocols collect multiple resting-state runs or sessions per subject.
Averaging at the subject level before group analysis has two benefits:

1. **Reduces within-subject noise** — more data per estimate
2. **Keeps the group model simple** — one observation per subject, so
   standard GLM exchangeability holds

The alternative (mixed-effects model with runs nested in subjects) is more
powerful but requires more complex permutation schemes not covered here.

## 1.1 Expected Directory Structure

```
derivatives/
├── sub-01/
│   ├── ses-01/
│   │   └── sub-01_ses-01_run-01_fc.csv
│   │   └── sub-01_ses-01_run-02_fc.csv
│   └── ses-02/
│       └── sub-01_ses-02_run-01_fc.csv
├── sub-02/
│   └── ...
└── subject_level/          ← aggregated outputs go here
    ├── sub-01_fc.csv
    └── sub-02_fc.csv
```

If your data has only one run per subject, skip directly to Section 1.3.

## 1.2 Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DERIVATIVES_DIR   = '/path/to/derivatives'
SUBJECT_LEVEL_DIR = '/path/to/derivatives/subject_level'
os.makedirs(SUBJECT_LEVEL_DIR, exist_ok=True)

# ── Subject list ──────────────────────────────────────────────────────────────
# List all subject IDs to process
SUBJECTS = ['sub-01', 'sub-', 'sub-03']   # replace with your subjects


# ── Run discovery pattern ─────────────────────────────────────────────────────
# Adjust the glob pattern to match your file naming convention.
# Examples:
#   Multiple runs, one session : 'sub-{sub}/sub-{sub}_run-*_fc.csv'
#   Multiple sessions          : 'sub-{sub}/ses-*/sub-{sub}_ses-*_fc.csv'
#   Flat directory             : 'sub-{sub}_*_fc.csv'
RUN_GLOB = 'sub-{sub}/sub-{sub}_run-*_fc.csv'   # edit me

# ── Output filename pattern ───────────────────────────────────────────────────
# Subject-level aggregated FC matrices
SUBJECT_FC_PATTERN = '{sub}.csv'   # e.g. sub-01_fc.csv

## 1.3 Aggregate Runs → Subject-Level FC

**Function:** `aggregate_subject_runs(run_paths, method='mean')`

Reads each run's FC matrix CSV and averages element-wise across runs.
If a subject has only one run the result is identical to the input.

> **Fisher-z note:** `aggregate_subject_runs()` works directly on Pearson r
> values. For a more statistically rigorous average, Fisher-z transform the
> matrices before calling this function, then invert: `np.tanh(fc_mean)`.

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# For each subject:
#   1. Glob for all run-level FC CSVs using RUN_GLOB
#   2. Call aggregate_subject_runs(run_paths)
#   3. Call save_subject_fc(fc_mean, labels, output_path)
#
# Template:
# for sub in SUBJECTS:
#     run_paths = sorted(glob.glob(
#         os.path.join(DERIVATIVES_DIR, RUN_GLOB.format(sub=sub))))
#     print(f'{sub}: {len(run_paths)} run(s) found')
#     fc_mean, labels = aggregate_subject_runs(run_paths)
#     out = os.path.join(SUBJECT_LEVEL_DIR, SUBJECT_FC_PATTERN.format(sub=sub))
#     save_subject_fc(fc_mean, labels, out)


# ─────────────────────────────────────────────────────────────────────────────
# After running, verify:
# subject_csvs = sorted(glob.glob(os.path.join(SUBJECT_LEVEL_DIR, '*_fc.csv')))
# print(f'Subject-level files written: {len(subject_csvs)}')

## 1.4 Quick Sanity Check — Group Mean

Before running statistics, always visualise the group mean FC.
It should show strong within-network block structure (diagonal blocks).

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# 1. Load all subject-level CSVs using load_fc_from_file_list() or a glob loop
# 2. Compute group mean: mean_mat = matrices.mean(axis=0)
# 3. Visualise with plot_fc_matrix() from fc_helpers


# ─────────────────────────────────────────────────────────────────────────────
plt.show()

---
# Section 2 — Permutation-Based Statistics

## Overview

We use a **mass-univariate GLM** — the same model at every edge simultaneously.
Significance is assessed by a permutation test rather than t-distribution
tables, so no distributional assumptions are required.

| Step | What happens |
|------|-------------|
| 1. Fit GLM | Compute t-statistic at every edge using real data and labels |
| 2. Permute | Shuffle design matrix rows (or flip signs) 5000× |
| 3. Record max |t| | Build the null distribution of the maximum statistic |
| 4. Threshold | Compare each observed t to the null to get FWE p-values |
| 5. FDR | Apply Benjamini–Hochberg to parametric p-values |

## 2.1 Input File Formats

### A. FC matrix file list (`fc_inputs.txt`)

One absolute path per line. Lines starting with `#` are ignored.

```
# Group-level FC input list
# Generated: 2026-05-15
#
# Controls
/data/derivatives/subject_level/sub-01_fc.csv
/data/derivatives/subject_level/sub-02_fc.csv
/data/derivatives/subject_level/sub-03_fc.csv
# Patients
/data/derivatives/subject_level/sub-04_fc.csv
/data/derivatives/subject_level/sub-05_fc.csv
```

---

### B. Design matrix (`design_matrix.csv`)

One row per subject, one column per regressor.
The `subject_id` column is optional but strongly recommended — it prevents
silent row-order mismatches between the file list and the design matrix.

**One-sample t-test** — test whether group mean FC ≠ 0
Contrast: `[1]`
```
subject_id,intercept
sub-01,1
sub-02,1
sub-03,1
sub-04,1
sub-05,1
```

**Two-sample t-test** — patients (1) vs controls (0)
Contrast: `[0, 1]`
```
subject_id,intercept,group
sub-01,1,0
sub-02,1,0
sub-03,1,0
sub-04,1,1
sub-05,1,1
```

**Continuous covariate / regression** — e.g. symptom score
Contrast: `[0, 1]`  (test the slope, not the intercept)
```
subject_id,intercept,symptom_z
sub-01,1,-1.23
sub-02,1,-0.45
sub-03,1, 0.12
sub-04,1, 0.78
sub-05,1, 1.52
```

**ANCOVA — group difference controlling for age and motion**
Contrast: `[0, 1, 0, 0]`
```
subject_id,intercept,group,age_z,fd_mean_z
sub-01,1,0,-1.10, 0.22
sub-02,1,0,-0.30, 0.05
sub-03,1,0, 0.40,-0.18
sub-04,1,1, 0.55, 0.31
sub-05,1,1, 1.20,-0.12
```

> **Tip:** Always z-score continuous covariates (mean 0, SD 1) so the
> intercept represents the group mean at the average covariate value.

## 2.2 Configuration

In [ ]:
# ── Input files ──────────────────────────────────────────────────────────────
FC_LIST_TXT      = '/path/to/fc_inputs.txt'       # one FC matrix path per line
DESIGN_MATRIX_CSV = '/path/to/design_matrix.csv'  # subjects × regressors

# ── Contrast vector ───────────────────────────────────────────────────────────
# Must match the columns of your design matrix.
# Examples:
#   One-sample (intercept only)  : CONTRAST = [1]
#   Two-sample (group column)    : CONTRAST = [0, 1]
#   Covariate slope              : CONTRAST = [0, 1]
#   ANCOVA group effect          : CONTRAST = [0, 1, 0, 0]
CONTRAST = [1]   # edit me

# ── Permutation settings ──────────────────────────────────────────────────────
N_PERM = 5000      # use 500 for fast testing, 5000+ for analysis
ALPHA  = 0.05

# ── Output directory ─────────────────────────────────────────────────────────
OUTPUT_DIR = './group_analysis_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2.3 Load FC Matrices and Design Matrix

**`load_fc_from_file_list(txt_path)`** reads the text file and stacks all
matrices into a 3D array. It verifies that parcel labels are identical
across subjects and raises an error immediately on mismatch.

**`load_design_matrix(csv_path, subject_ids)`** reads the design matrix CSV
and reorders rows to match the subject order from the file list.

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# 1. Load matrices:   matrices, labels, subject_ids = load_fc_from_file_list(...)
# 2. Load design:     X, col_names = load_design_matrix(..., subject_ids=...)
# 3. Print shapes to confirm alignment:
#    print(matrices.shape)   # (n_subjects, N, N)
#    print(X.shape)          # (n_subjects, n_regressors)


# ─────────────────────────────────────────────────────────────────────────────

## 2.4 Run the Permutation GLM

**`permutation_glm(matrices, X, contrast, n_perm, alpha)`**

The function automatically selects the permutation scheme:
- **`sign_flip`** — when X is intercept-only (one-sample test)
- **`shuffle`** — for all other designs (two-sample, regression)

You can override with `perm_method='shuffle'` or `perm_method='sign_flip'`.

> **Runtime estimate:** ~30 s for 200 parcels, 30 subjects, 5000 permutations
> on a modern laptop (vectorised across edges).

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# results = permutation_glm(
#     matrices   = matrices,
#     design_matrix = X,
#     contrast   = CONTRAST,
#     n_perm     = N_PERM,
#     alpha      = ALPHA,
# )


# ─────────────────────────────────────────────────────────────────────────────
# After running:
# print(f'Max |t|        : {np.abs(results["t_mat"]).max():.3f}')
# print(f'FWE threshold  : {np.percentile(results["null_max_t"], 95):.3f}')

## 2.5 Design Limitations — Read Before Interpreting Results

The permutation GLM implemented here makes the following assumptions.
Violating them can produce invalid p-values.

| Design | Status | Notes |
|--------|--------|-------|
| One-sample (intercept only) | ✅ Valid | Uses sign-flip; assumes symmetry around 0 under H₀ |
| Two-sample (group 0/1) | ✅ Valid | Assumes exchangeability across groups |
| Continuous regression | ✅ Valid | Row-shuffle breaks X–Y correlation under H₀ |
| ANCOVA (group + nuisance) | ⚠️ Approximate | Nuisance not properly controlled — use Freedman–Lane |
| Repeated measures / paired | ❌ Invalid | Row-shuffle breaks pairing structure |
| Hierarchical (multi-site) | ❌ Invalid | Subjects from different sites are not exchangeable |
| Very small n (< 8/group) | ❌ Underpowered | Permutation space too small for p < 0.05 |

**Freedman–Lane for designs with nuisance covariates:**
1. Regress nuisance regressors out of Y → residuals R
2. Permute R
3. Add fitted nuisance back: Y_perm = X_nuisance @ beta_nuisance + R_perm
4. Fit full model on Y_perm

This procedure is not implemented here. For publication-quality ANCOVA
results, consider `permuco` (R) or `FSL randomise`.

---
# Section 3 — Corrected Maps and Saving Results

## When to use each correction

| Correction | Controls | Power | Use when |
|------------|----------|-------|----------|
| **Uncorrected** | Nothing — inflated FP rate | Highest | Exploratory only; never for inference |
| **FDR (BH)** | Expected % false discoveries | Moderate | Standard choice for FC analysis |
| **FWE (max-t)** | Probability of any false positive | Lowest | When any false positive is unacceptable |

Always report all three. A result that survives FWE is the strongest claim.
A result that only survives FDR is still valid but should be interpreted with
more caution. If nothing survives FDR, do not interpret the uncorrected map.

## 3.1 View the Corrected Maps

**`plot_corrected_maps(results, labels, alpha)`**

Four-panel figure: t-statistic | uncorrected | FDR | FWE.
Non-significant edges are zeroed out in the thresholded panels.

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# fig = plot_corrected_maps(
#     results    = results,
#     labels     = labels,   # pass None if N > 20 (too many ticks)
#     alpha      = ALPHA,
#     output_path = os.path.join(OUTPUT_DIR, 'corrected_maps.png'),
# )


# ─────────────────────────────────────────────────────────────────────────────
plt.show()

## 3.2 Null Distribution

**`plot_null_distribution(results, alpha)`**

Inspect the permutation null distribution of max|t|.
The distribution should be smooth and approximately bell-shaped.
A ragged or multi-modal null suggests n_perm is too small or the
permutation scheme is inappropriate for your design.

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# fig = plot_null_distribution(
#     results     = results,
#     alpha       = ALPHA,
#     output_path = os.path.join(OUTPUT_DIR, 'null_distribution.png'),
# )


# ─────────────────────────────────────────────────────────────────────────────
plt.show()

## 3.3 Save Edge-Level Results

**`save_edge_results(results, labels, output_path)`**

Writes a CSV with one row per unique edge (upper triangle only) containing
t, p_unc, p_fdr, p_fwe, and significance flags.
Use this file for downstream analyses: effect-size filtering, correlation
with behavioural variables, or chord/glass-brain visualisation.

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# edge_df = save_edge_results(
#     results     = results,
#     labels      = labels,
#     output_path = os.path.join(OUTPUT_DIR, 'edge_results.csv'),
# )


# ─────────────────────────────────────────────────────────────────────────────
# Inspect strongest effects:
# edge_df[edge_df.sig_fdr].sort_values('t', ascending=False).head(20)

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
# Run after results and edge_df are defined above.

# summary = {
#     'n_subjects'         : matrices.shape[0],
#     'n_parcels'          : matrices.shape[1],
#     'n_edges'            : len(edge_df),
#     'perm_method'        : results['perm_method'],
#     'n_perm'             : len(results['null_max_t']),
#     'fwe_threshold_t'    : round(float(np.percentile(results['null_max_t'], 95)), 3),
#     'n_sig_uncorrected'  : int(edge_df.sig_unc.sum()),
#     'n_sig_fdr'          : int(edge_df.sig_fdr.sum()),
#     'n_sig_fwe'          : int(edge_df.sig_fwe.sum()),
# }
# for k, v in summary.items():
#     print(f'{k:<25}: {v}')

---
# ✅ Answer Key

> Scroll past this line only after completing the exercise above.

DERIVATIVES_DIR### AK-1. Aggregate runs and save subject-level FC

In [ ]:
# ── ANSWER KEY: Section 1.3 ──────────────────────────────────────────────────
for sub in SUBJECTS:
    run_paths = sorted(glob.glob(
        os.path.join(DERIVATIVES_DIR, RUN_GLOB.format(sub=sub))))
    print(f'{sub}: {len(run_paths)} run(s)')
    if len(run_paths) == 0:
        print(f'  WARNING: no run files found — check RUN_GLOB pattern')
        continue
    fc_mean, labels = aggregate_subject_runs(run_paths)
    out = os.path.join(SUBJECT_LEVEL_DIR, SUBJECT_FC_PATTERN.format(sub=sub))
    save_subject_fc(fc_mean, labels, out)

subject_csvs = sorted(
    glob.glob(os.path.join(SUBJECT_LEVEL_DIR, 'sub-*.csv'))
)

# write file list to text file
txt_out = os.path.join(SUBJECT_LEVEL_DIR, 'subject_csvs.txt')

with open(txt_out, 'w') as f:
    for csv_file in subject_csvs:
        f.write(csv_file + '\n')

print(f'Subject-level files written: {len(subject_csvs)}')
print(f'File list saved to: {txt_out}')

### AK-2. Sanity check — group mean FC

In [ ]:
# ── ANSWER KEY: Section 1.4 ──────────────────────────────────────────────────
_mats, _labels, _sids = load_fc_from_file_list(FC_LIST_TXT)
mean_mat = _mats.mean(axis=0)

fig = plot_fc_matrix(
    mean_mat, labels=_labels,
    title=f'Group mean FC  (n = {_mats.shape[0]})',
    vmin=-1, vmax=1, figsize=(10, 9),
)
fig.savefig(os.path.join(OUTPUT_DIR, 'group_mean_fc.png'), dpi=150, bbox_inches='tight')
plt.show()

subject_ids### AK-3. Load data and design matrix

In [ ]:
# ── ANSWER KEY: Section 2.3 ──────────────────────────────────────────────────
matrices, labels, subject_ids = load_fc_from_file_list(FC_LIST_TXT)
X, col_names = load_design_matrix(DESIGN_MATRIX_CSV, subject_ids=subject_ids)

print(f'\nMatrices : {matrices.shape}    (subjects × parcels × parcels)')
print(f'Design X : {X.shape}    columns = {col_names}')
print(f'Contrast : {CONTRAST}')

### AK-4. Run permutation GLM

In [ ]:
# ── ANSWER KEY: Section 2.4 ──────────────────────────────────────────────────
results = permutation_glm(
    matrices      = matrices,
    design_matrix = X,
    contrast      = CONTRAST,
    n_perm        = N_PERM,
    alpha         = ALPHA,
)

print(f'\nMax |t|       : {np.abs(results["t_mat"]).max():.3f}')
print(f'FWE threshold : {np.percentile(results["null_max_t"], 95):.3f}  (95th pct null max|t|)')

### AK-5. Corrected maps

In [ ]:
# ── ANSWER KEY: Section 3.1 ──────────────────────────────────────────────────
fig = plot_corrected_maps(
    results     = results,
    labels      = labels if len(labels) <= 20 else None,
    alpha       = ALPHA,
    output_path = os.path.join(OUTPUT_DIR, 'corrected_maps.png'),
)
plt.show()

### AK-6. Null distribution

In [ ]:
# ── ANSWER KEY: Section 3.2 ──────────────────────────────────────────────────
fig = plot_null_distribution(
    results     = results,
    alpha       = ALPHA,
    output_path = os.path.join(OUTPUT_DIR, 'null_distribution.png'),
)
plt.show()

### AK-7. Save edge results and print summary

In [ ]:
# ── ANSWER KEY: Section 3.3 ──────────────────────────────────────────────────
edge_df = save_edge_results(
    results     = results,
    labels      = labels,
    output_path = os.path.join(OUTPUT_DIR, 'edge_results.csv'),
)

print('\n── Top 10 edges by |t| (FDR significant) ──')
top = edge_df[edge_df.sig_fdr].reindex(
    edge_df[edge_df.sig_fdr]['t'].abs().sort_values(ascending=False).index
)
print(top[['roi_i','roi_j','t','p_fdr','p_fwe','sig_fwe']].head(10).to_string(index=False))

print('\n── Summary ──')
summary = {
    'n_subjects'        : int(matrices.shape[0]),
    'n_parcels'         : int(matrices.shape[1]),
    'n_edges_tested'    : int(len(edge_df)),
    'perm_method'       : results['perm_method'],
    'n_perm'            : int(len(results['null_max_t'])),
    'fwe_crit_t'        : round(float(np.percentile(results['null_max_t'], 95)), 3),
    'n_sig_uncorrected' : int(edge_df.sig_unc.sum()),
    'n_sig_fdr'         : int(edge_df.sig_fdr.sum()),
    'n_sig_fwe'         : int(edge_df.sig_fwe.sum()),
}
for k, v in summary.items():
    print(f'  {k:<22}: {v}')

pd.DataFrame([summary]).to_csv(os.path.join(OUTPUT_DIR, 'analysis_summary.csv'), index=False)
print(f'\nAll outputs written to: {OUTPUT_DIR}')